
# Optimizer r04 — linearized route–time bus allocation (Gurobi)

This notebook builds a **clean MILP implementation** of the linearized OC Transpo formulation using the modular files in `src/`:

- `src.demand_linear.LinearDemandModel`
- `src.costs.CostModel`
- `src.benefits_linear.LinearBenefitModel`
- `src.constraints.*`

The intent is to keep the notebook focused on:
1. loading the new dataset,
2. constructing coefficient dictionaries,
3. instantiating the model objects from `src/`,
4. building a compact Gurobi model, and
5. returning tidy solution tables for analysis.


In [1]:

from pathlib import Path
import math
import sys

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')


In [ ]:

# Make sure project imports resolve when this notebook is run either
# from the repository root or from a copied/exported location.

project_root_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/mnt/data'),
]

for root in project_root_candidates:
    src_path = root / 'src'
    if src_path.exists() and str(root) not in sys.path:
        sys.path.insert(0, str(root))
        break

try:
    import gurobipy as gp
    from gurobipy import GRB, quicksum
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        'gurobipy is not installed in this environment. Run this notebook in your '
        'local MECH 5800 project environment where Gurobi is configured.'
    ) from e

from src.demand_linear import LinearDemandModel
from src.costs import CostModel
from src.benefits_linear import LinearBenefitModel
from src.constraints import (
    add_integer_bus_vars,
    add_budget_constraint,
    add_fleet_constraints,
)



## 1. Load the new linearized dataset

The route–time dataset is the main model input. The separate time-block parameter CSV is kept available for sensitivity work, but the route-level subset already contains the parameters needed by the current `src/` classes.


In [3]:

def first_existing_path(candidates):
    path = next((Path(p) for p in candidates if Path(p).exists()), None)
    if path is None:
        raise FileNotFoundError(f'Could not find any of: {candidates}')
    return path

route_data_path = first_existing_path([
    'data/route_timeblock_subset_v01.csv',
    'route_timeblock_subset_v01.csv',
    '/mnt/data/route_timeblock_subset_v01.csv',
])

param_data_path = first_existing_path([
    'data/timeblock_parameters_v01.csv',
    'timeblock_parameters_v01.csv',
    '/mnt/data/timeblock_parameters_v01.csv',
])

raw_df = pd.read_csv(route_data_path)
param_df = pd.read_csv(param_data_path)

print(f'Using route-time data: {route_data_path}')
print(f'Using time-block parameters: {param_data_path}')
raw_df.head()


Using route-time data: data\route_timeblock_subset_v01.csv
Using time-block parameters: data\timeblock_parameters_v01.csv


,Route (r),Time Block (t),Route type,"Old ridership (x_old,r,t)",Trip length km (L_r),"Average length hr (T_r,t)","Old frequency of buses (f_old,r,t)","Old number of buses (n_old,r,t)","Time block hr (H_block,t)",Driver horuly wage rate $/hr (W_driver),Price of fuel $/litre (P_fuel),Fuel consumption (FC),Maintenace cost $/km (P_maintenance),Passenger hourly wage rate $/hr (W_passenger),VTTS fraction of passenger wage F_wage,"Emission intensity of a car kgCO2eq/KPT (I_GHG,car)",Average Length of passenger trip (L_avg_trip),Social cost of carbon $/tonne CO2eq (SCC)
0,5,AM Peak,Frequent,292,9.700,1.050,7.500,8,2.500,34.040,1.300,0.242,0.400,35.060,1.000,0.330,1.940,275
1,5,Early AM,Frequent,23,9.700,0.920,2.000,2,3.500,34.040,1.300,0.198,0.400,35.060,0.700,0.295,1.940,275
2,5,Evening,Frequent,162,9.700,0.980,3.000,3,4.500,34.040,1.300,0.220,0.400,35.060,0.700,0.300,1.940,275
3,5,Midday,Frequent,566,9.700,0.980,5.000,5,6.000,34.040,1.300,0.220,0.400,35.060,0.700,0.300,1.940,275
4,5,Night,Frequent,73,9.700,0.860,2.000,2,4.000,34.040,1.300,0.198,0.400,35.060,0.700,0.295,1.940,275


In [4]:

# Standardize the route-time dataset to concise internal names.

col_map = {
    'Route (r)': 'route',
    'Time Block (t)': 'time_block',
    'Route type': 'route_type',
    'Old ridership (x_old,r,t)': 'x_old',
    'Trip length km (L_r)': 'L_r',
    'Average length hr (T_r,t)': 'T_rt',
    'Old frequency of buses (f_old,r,t)': 'f_old',
    'Old number of buses (n_old,r,t)': 'n_old',
    'Time block hr (H_block,t)': 'H_block',
    'Driver horuly wage rate $/hr (W_driver)': 'W_driver',
    'Price of fuel $/litre (P_fuel)': 'P_fuel',
    'Fuel consumption (FC)': 'fuel_consumption',
    'Maintenace cost $/km (P_maintenance)': 'P_maintenance',
    'Passenger hourly wage rate $/hr (W_passenger)': 'W_passenger',
    'VTTS fraction of passenger wage F_wage': 'F_wage',
    'Emission intensity of a car kgCO2eq/KPT (I_GHG,car)': 'I_GHG_car',
    'Average Length of passenger trip (L_avg_trip)': 'L_avg_trip',
    'Social cost of carbon $/tonne CO2eq (SCC)': 'SCC',
}

df = raw_df.rename(columns=col_map).copy()
df['route'] = df['route'].astype(str)
df['time_block'] = df['time_block'].astype(str)
df['key'] = list(zip(df['route'], df['time_block']))

df[['route', 'time_block', 'route_type', 'x_old', 'n_old', 'T_rt', 'L_r']].head()


,route,time_block,route_type,x_old,n_old,T_rt,L_r
0,5,AM Peak,Frequent,292,8,1.050,9.700
1,5,Early AM,Frequent,23,2,0.920,9.700
2,5,Evening,Frequent,162,3,0.980,9.700
3,5,Midday,Frequent,566,5,0.980,9.700
4,5,Night,Frequent,73,2,0.860,9.700



## 2. Scenario assumptions for the linearized MILP

The `src/` files define the algebra, but we still need a few scenario choices for the optimization run:

- how far each route–time service level may move from current service,
- how much total fleet is available in each time block, and
- the total incremental operating budget for the subset.

These are grouped here so they are easy to tune.


In [5]:

# --- service adjustment bounds ---
# Frequent routes are allowed to move a bit more than local routes.
max_change_by_route_type = {
    'Frequent': 2,
    'Local': 1,
}

# --- block-level fleet expansion above the current subset allocation ---
extra_buses_by_block = {
    'Early AM': 2,
    'AM Peak': 3,
    'Midday': 3,
    'PM Peak': 3,
    'Evening': 2,
    'Night': 2,
}

# --- incremental budget for this subset model ---
# CostModel returns change in operating cost relative to current service,
# so this is an allowable *increase* in daily operating cost.
budget_total = 3_000.0

# --- linearization control ---
# Based on the corrected sign convention that ridership rises with additional service.
# If x ~ x_old * (n_new / n_old)^elasticity, then dx/dn at n_old is elasticity * x_old / n_old.
demand_elasticity = 0.5



## 3. Build coefficient dictionaries for the `src/` models

For this linearized notebook, the coefficient construction is kept explicit:

- **Demand slope**
  \[
  lpha_{r,t} pprox 
arepsilon 
rac{x^{old}_{r,t}}{n^{old}_{r,t}}
  \]
- **Time benefit per added bus** from a linearization of average waiting time
  \[
  eta^{time}_{r,t} pprox x^{old}_{r,t} \cdot W_{pass} \cdot F_{wage} \cdot 
rac{0.5 T_{r,t}}{(n^{old}_{r,t})^2}
  \]
- **Emission benefit per added bus** from induced ridership shifted from car travel
  \[
  eta^{em}_{r,t} pprox lpha_{r,t} \cdot L^{avg}_{trip} \cdot I_{GHG,car} \cdot 
rac{SCC}{1000}
  \]

If you later want coefficients taken directly from the report or from Jessica's latest calibration, this is the one cell to swap out.


In [6]:

R = sorted(df['route'].unique())
T = ['Early AM', 'AM Peak', 'Midday', 'PM Peak', 'Evening', 'Night']
RT = [(r, t) for r in R for t in T]

# Quick integrity check
missing = set(RT) - set(df['key'])
if missing:
    raise ValueError(f'Missing route-time combinations in dataset: {sorted(missing)}')

# Dictionaries keyed by (route, time_block)
x_old = df.set_index('key')['x_old'].to_dict()
n_old = df.set_index('key')['n_old'].astype(int).to_dict()
T_rt = df.set_index('key')['T_rt'].to_dict()
route_type = df.set_index('key')['route_type'].to_dict()

# Dictionaries keyed by route or by time block
L_r = df.groupby('route')['L_r'].first().to_dict()
H_block = df.groupby('time_block')['H_block'].first().to_dict()

# Scalars that are currently constant across the subset
W_driver = float(df['W_driver'].iloc[0])
P_maintenance = float(df['P_maintenance'].iloc[0])

# Time-varying fuel consumption is present in the dataset, but CostModel currently
# expects a scalar. Use the dataset-average value for now so the notebook is fully
# compatible with the existing src/costs.py interface.
fuel_consumption = float(df['fuel_consumption'].mean())
P_fuel = float(df['P_fuel'].iloc[0])

alpha = {}
beta_time = {}
beta_emissions = {}
n_min = {}
n_max = {}

for _, row in df.iterrows():
    k = row['key']
    n0 = int(row['n_old'])
    x0 = float(row['x_old'])
    T0 = float(row['T_rt'])

    alpha[k] = demand_elasticity * x0 / n0
    beta_time[k] = x0 * row['W_passenger'] * row['F_wage'] * (0.5 * T0 / (n0 ** 2))
    beta_emissions[k] = alpha[k] * row['L_avg_trip'] * row['I_GHG_car'] * row['SCC'] / 1000.0

    delta_cap = max_change_by_route_type[row['route_type']]
    n_min[k] = max(0, n0 - delta_cap)
    n_max[k] = n0 + delta_cap

fleet_cap = {
    t: int(df.loc[df['time_block'] == t, 'n_old'].sum() + extra_buses_by_block[t])
    for t in T
}

coef_df = pd.DataFrame({
    'route': [k[0] for k in RT],
    'time_block': [k[1] for k in RT],
    'x_old': [x_old[k] for k in RT],
    'n_old': [n_old[k] for k in RT],
    'alpha': [alpha[k] for k in RT],
    'beta_time': [beta_time[k] for k in RT],
    'beta_emissions': [beta_emissions[k] for k in RT],
    'n_min': [n_min[k] for k in RT],
    'n_max': [n_max[k] for k in RT],
})

coef_df.head(12)


,route,time_block,x_old,n_old,alpha,beta_time,beta_emissions,n_min,n_max
0,5,Early AM,23,2,5.750,64.914,0.905,0,4
1,5,AM Peak,292,8,18.250,83.980,3.213,6,10
2,5,Midday,566,5,56.600,272.259,9.059,3,7
3,5,PM Peak,303,8,18.938,87.143,3.334,6,10
4,5,Evening,162,3,27.000,216.460,4.321,1,5
5,5,Night,73,2,18.250,192.593,2.872,0,4
6,6,Early AM,177,3,29.500,359.581,8.185,1,5
7,6,AM Peak,1724,13,66.308,307.582,20.580,11,15
8,6,Midday,3513,8,219.562,"1,070.965",61.950,6,10
9,6,PM Peak,2781,13,106.962,496.163,33.197,11,15


## 4. Instantiate the modular `src/` model objects

In [7]:

demand_model = LinearDemandModel(
    alpha=alpha,
    n_old=n_old,
    x_old=x_old,
)

cost_model = CostModel(
    H_block=H_block,
    W_driver=W_driver,
    L_r=L_r,
    T_rt=T_rt,
    P_fuel=P_fuel,
    fuel_consumption=fuel_consumption,
    P_maintenance=P_maintenance,
    n_old=n_old,
)

benefit_model = LinearBenefitModel(
    beta_time=beta_time,
    beta_emissions=beta_emissions,
    n_old=n_old,
)



## 5. Build the Gurobi model

Objective:
\[
\min \sum_{r,t} \left(\Delta C_{r,t} - \Delta B_{r,t}
ight)
\]

Constraints used here:
- integer bus counts,
- per-block fleet caps,
- total incremental operating budget,
- variable bounds around current service.


In [8]:
m = gp.Model('oc_transpo_linearized_r04')
m.Params.OutputFlag = 1

n_new = add_integer_bus_vars(
    model=m,
    RT=RT,
    n_min=n_min,
    n_max=n_max,
    name='n_new',
)

objective = quicksum(
    cost_model.total(r, t, n_new[r, t]) - benefit_model.benefit_total(r, t, n_new[r, t])
    for r, t in RT
)
m.setObjective(objective, GRB.MINIMIZE)


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2796207
Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Set parameter OutputFlag to value 1


In [9]:

# src.constraints.add_fleet_constraints currently expects a scalar fleet cap,
# but this project needs a time-block-specific fleet limit.
# Build those block constraints cleanly here.

fleet_constraints = {
    t: m.addConstr(
        quicksum(n_new[r, t] for r in R) <= fleet_cap[t],
        name=f'fleet[{t}]'
    )
    for t in T
}

budget_constraint = add_budget_constraint(
    model=m,
    n_new=n_new,
    RT=RT,
    cost_model=cost_model,
    budget_total=budget_total,
)

m.optimize()

print('Status code:', m.Status)
if m.Status == GRB.OPTIMAL:
    print('Optimal objective:', m.ObjVal)
    print('MIP gap:', m.MIPGap)
    print('Runtime (s):', m.Runtime)


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 10.0 (19045.2))

CPU model: 11th Gen Intel(R) Core(TM) i7-1195G7 @ 2.90GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Academic license 2796207 - for non-commercial use only - registered to jo___@cmail.carleton.ca
Optimize a model with 7 rows, 24 columns and 48 nonzeros
Model fingerprint: 0x9296732e
Variable types: 0 continuous, 24 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [1e+01, 9e+02]
  Bounds range     [1e+00, 2e+01]
  RHS range        [1e+01, 2e+04]
Found heuristic solution: objective -6136.395288
Presolve removed 3 rows and 4 columns
Presolve time: 0.02s
Presolved: 4 rows, 20 columns, 32 nonzeros
Variable types: 0 continuous, 20 integer (0 binary)

Root relaxation: objective -1.414766e+04, 3 iterations, 0.01 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds

## 6. Collect route–time results

In [10]:

if m.Status != GRB.OPTIMAL:
    raise RuntimeError(f'Model did not solve to optimality. Status = {m.Status}')

rows = []
for r, t in RT:
    n_star = int(round(n_new[r, t].X))
    rows.append({
        'route': r,
        'time_block': t,
        'route_type': route_type[(r, t)],
        'n_old': n_old[(r, t)],
        'n_new': n_star,
        'delta_n': n_star - n_old[(r, t)],
        'x_old': x_old[(r, t)],
        'x_new_linear': demand_model.ridership(r, t, n_star),
        'delta_ridership': demand_model.delta_ridership(r, t, n_star),
        'operating_cost_delta': cost_model.total(r, t, n_star),
        'benefit_time_delta': benefit_model.benefit_time(r, t, n_star),
        'benefit_emissions_delta': benefit_model.benefit_emissions(r, t, n_star),
        'benefit_total_delta': benefit_model.benefit_total(r, t, n_star),
        'net_objective_contribution': cost_model.total(r, t, n_star) - benefit_model.benefit_total(r, t, n_star),
        'at_lower_bound': n_star == n_min[(r, t)],
        'at_upper_bound': n_star == n_max[(r, t)],
    })

solution_df = pd.DataFrame(rows)
solution_df


,route,time_block,route_type,n_old,n_new,delta_n,x_old,x_new_linear,delta_ridership,operating_cost_delta,benefit_time_delta,benefit_emissions_delta,benefit_total_delta,net_objective_contribution,at_lower_bound,at_upper_bound
0,5,Early AM,Frequent,2,0,-2,23,11.500,-11.500,-288.910,-129.827,-1.810,-131.637,-157.273,True,False
1,5,AM Peak,Frequent,8,6,-2,292,255.500,-36.500,-201.887,-167.959,-6.426,-174.385,-27.501,True,False
2,5,Midday,Frequent,5,3,-2,566,452.800,-113.200,-489.960,-544.518,-18.118,-562.636,72.676,True,False
3,5,PM Peak,Frequent,8,6,-2,303,265.125,-37.875,-282.641,-174.287,-6.668,-180.955,-101.687,True,False
4,5,Evening,Frequent,3,1,-2,162,108.000,-54.000,-367.470,-432.921,-8.643,-441.564,74.094,True,False
5,5,Night,Frequent,2,0,-2,73,36.500,-36.500,-334.220,-385.187,-5.744,-390.931,56.712,True,False
6,6,Early AM,Frequent,3,5,2,177,236.000,59.000,293.390,719.162,16.369,735.532,-442.142,False,True
7,6,AM Peak,Frequent,13,15,2,1724,"1,856.615",132.615,204.301,615.164,41.159,656.323,-452.023,False,True
8,6,Midday,Frequent,8,10,2,3513,"3,952.125",439.125,497.013,"2,141.930",123.899,"2,265.829","-1,768.816",False,True
9,6,PM Peak,Frequent,13,15,2,2781,"2,994.923",213.923,286.021,992.327,66.394,"1,058.721",-772.700,False,True


In [11]:

summary = {
    'objective_value': m.ObjVal,
    'budget_total': budget_total,
    'budget_used': solution_df['operating_cost_delta'].sum(),
    'budget_slack': budget_total - solution_df['operating_cost_delta'].sum(),
    'total_delta_ridership': solution_df['delta_ridership'].sum(),
    'total_cost_delta': solution_df['operating_cost_delta'].sum(),
    'total_time_benefit_delta': solution_df['benefit_time_delta'].sum(),
    'total_emissions_benefit_delta': solution_df['benefit_emissions_delta'].sum(),
    'total_benefit_delta': solution_df['benefit_total_delta'].sum(),
    'vars_at_lower_bound': int(solution_df['at_lower_bound'].sum()),
    'vars_at_upper_bound': int(solution_df['at_upper_bound'].sum()),
    'runtime_seconds': m.Runtime,
    'node_count': m.NodeCount,
    'mip_gap': m.MIPGap,
}

for t in T:
    used = int(solution_df.loc[solution_df['time_block'] == t, 'n_new'].sum())
    summary[f'fleet_used[{t}]'] = used
    summary[f'fleet_cap[{t}]'] = fleet_cap[t]
    summary[f'fleet_slack[{t}]'] = fleet_cap[t] - used

pd.Series(summary)


objective_value                 -14,147.656
budget_total                      3,000.000
budget_used                       2,064.921
budget_slack                        935.079
total_delta_ridership             2,213.396
total_cost_delta                  2,064.921
total_time_benefit_delta         15,558.999
total_emissions_benefit_delta       653.578
total_benefit_delta              16,212.577
vars_at_lower_bound                   7.000
vars_at_upper_bound                  15.000
runtime_seconds                       0.057
node_count                            1.000
mip_gap                               0.000
fleet_used[Early AM]                  9.000
fleet_cap[Early AM]                  12.000
fleet_slack[Early AM]                 3.000
fleet_used[AM Peak]                  41.000
fleet_cap[AM Peak]                   41.000
fleet_slack[AM Peak]                  0.000
fleet_used[Midday]                   27.000
fleet_cap[Midday]                    27.000
fleet_slack[Midday]             

In [12]:

# Compact route-time allocation table for report use
allocation_pivot = solution_df.pivot(index='route', columns='time_block', values='n_new')
allocation_pivot = allocation_pivot[['Early AM', 'AM Peak', 'Midday', 'PM Peak', 'Evening', 'Night']]
allocation_pivot


time_block,Early AM,AM Peak,Midday,PM Peak,Evening,Night
route,,,,,,
5,0,6,3,6,1,0
6,5,15,10,15,7,5
7,2,16,11,16,7,5
9,2,4,3,4,2,1


In [13]:

# A more diagnostic view, sorted by the strongest incentive to add service.
solution_df.sort_values(['net_objective_contribution', 'route', 'time_block']).reset_index(drop=True)


,route,time_block,route_type,n_old,n_new,delta_n,x_old,x_new_linear,delta_ridership,operating_cost_delta,benefit_time_delta,benefit_emissions_delta,benefit_total_delta,net_objective_contribution,at_lower_bound,at_upper_bound
0,7,Night,Frequent,3,5,2,550,733.333,183.333,339.954,"2,129.700",52.055,"2,181.755","-1,841.802",False,True
1,6,Midday,Frequent,8,10,2,3513,"3,952.125",439.125,497.013,"2,141.930",123.899,"2,265.829","-1,768.816",False,True
2,7,Evening,Frequent,5,7,2,1075,"1,290.000",215.000,372.645,"1,720.149",62.081,"1,782.230","-1,409.585",False,True
3,6,Night,Frequent,3,5,2,448,597.333,149.333,339.834,"1,698.088",41.432,"1,739.520","-1,399.686",False,True
4,7,Midday,Frequent,9,11,2,3533,"3,925.556",392.556,496.860,"1,744.842",113.350,"1,858.193","-1,361.332",False,True
5,6,Evening,Frequent,5,7,2,1064,"1,276.800",212.800,372.760,"1,660.767",60.042,"1,720.808","-1,348.049",False,True
6,9,Midday,Local,2,3,1,368,460.000,92.000,244.515,"1,049.907",13.814,"1,063.721",-819.206,False,True
7,6,PM Peak,Frequent,13,15,2,2781,"2,994.923",213.923,286.021,992.327,66.394,"1,058.721",-772.700,False,True
8,9,AM Peak,Local,3,4,1,383,446.833,63.833,100.706,745.999,10.543,756.542,-655.835,False,True
9,7,PM Peak,Frequent,14,16,2,2555,"2,737.500",182.500,286.027,804.377,57.967,862.343,-576.316,False,True



## Notes / next cleanup items

1. `src.constraints.add_fleet_constraints` currently assumes a **single scalar fleet cap**, but this project needs a **different cap for each time block**. The notebook handles that directly for now.
2. `src.costs.CostModel` currently takes a **scalar** `fuel_consumption`, while the dataset gives it by time block. The notebook uses the subset-average value to stay compatible with the present `src/` interface.
3. If you want full consistency with the final report, the first upgrades I would make are:
   - allow time-block-specific fleet caps inside `src.constraints`,
   - allow time-block-specific fuel consumption inside `src.costs`, and
   - replace the coefficient-construction cell with the report-calibrated linearization exactly as finalized.
